In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from scipy.ndimage import label


def extract_monthly_event_stats_fast(
    filepath: str,
    precip_var: str = "tp",
    time_dim: str = "valid_time",
    threshold_m: float = 0.001,
):
    """
    Extract monthly precipitation-event statistics from a NetCDF file
    without converting the full gridded daily field to a large tidy DataFrame.

    Parameters
    ----------
    filepath : str
        Path to NetCDF file.
    precip_var : str, default "tp"
        Name of precipitation variable in the dataset.
    time_dim : str, default "valid_time"
        Name of time dimension.
    threshold_m : float, default 0.001
        Wet-day threshold in meters.

    Returns
    -------
    monthly_df : pd.DataFrame
        Monthly dataframe with month-start DatetimeIndex and columns:
        - event_frequency
        - max_intensity_m
        - max_duration_days
    event_df : pd.DataFrame
        Event-level dataframe for further analysis.
    """

    # -----------------------------
    # Load data
    # -----------------------------
    ds = xr.open_dataset(filepath)
    precip = ds[precip_var]

    if time_dim not in precip.dims:
        raise ValueError(f"'{time_dim}' is not a dimension of '{precip_var}'.")

    # Infer spatial dims as everything except time
    spatial_dims = [d for d in precip.dims if d != time_dim]
    if len(spatial_dims) < 1:
        raise ValueError("Precipitation array must have at least one non-time dimension.")

    # Stack spatial dims into one point dimension
    precip_stacked = precip.stack(point=spatial_dims).transpose(time_dim, "point")

    # Extract arrays
    time_vals = pd.to_datetime(precip_stacked[time_dim].values)
    precip_vals = precip_stacked.values  # shape: (time, point)

    # Precompute month-start timestamps for each time step
    month_start = pd.to_datetime(time_vals).to_period("M").to_timestamp()

    # Recover coordinate arrays for event_df
    point_index = precip_stacked["point"].to_index()

    event_records = []

    n_time, n_point = precip_vals.shape

    for j in range(n_point):
        series = precip_vals[:, j]

        # Skip all-NaN points quickly
        if np.all(np.isnan(series)):
            continue

        # Treat NaNs as dry days
        wet = np.isfinite(series) & (series > threshold_m)

        if not np.any(wet):
            continue

        labels, n_events = label(wet)

        if n_events == 0:
            continue

        # Find event boundaries
        label_changes = np.diff(np.r_[0, labels, 0])
        starts = np.where(label_changes > 0)[0]
        ends = np.where(label_changes < 0)[0] - 1

        # Spatial coordinates for this point
        point_coords = point_index[j]
        if not isinstance(point_coords, tuple):
            point_coords = (point_coords,)

        coord_map = dict(zip(spatial_dims, point_coords))

        # Summarize each event
        for event_id, start_idx, end_idx in zip(range(1, n_events + 1), starts, ends):
            event_slice = slice(start_idx, end_idx + 1)
            event_precip = series[event_slice]
            event_times = time_vals[event_slice]

            # Assign event to the month of its start
            event_month = pd.Timestamp(event_times[0]).to_period("M").to_timestamp()

            record = {
                **coord_map,
                "event_id": event_id,
                "month_start": event_month,
                "start_time": event_times[0],
                "end_time": event_times[-1],
                "duration_days": end_idx - start_idx + 1,
                "tp_m": np.nansum(event_precip),
            }
            event_records.append(record)

    # -----------------------------
    # Build event dataframe
    # -----------------------------
    if len(event_records) == 0:
        monthly_df = pd.DataFrame(
            columns=["event_frequency", "max_intensity_m", "max_duration_days"]
        )
        monthly_df.index = pd.DatetimeIndex([], name="month_start")
        event_df = pd.DataFrame()
        return monthly_df, event_df

    event_df = pd.DataFrame(event_records).sort_values("month_start")

    # -----------------------------
    # Monthly summaries
    # -----------------------------
    monthly_frequency = event_df.groupby("month_start").size()
    monthly_frequency.name = "event_frequency"

    monthly_max_intensity = event_df.groupby("month_start")["tp_m"].max()
    monthly_max_intensity.name = "max_intensity_m"

    monthly_max_duration = event_df.groupby("month_start")["duration_days"].max()
    monthly_max_duration.name = "max_duration_days"

    monthly_df = pd.concat(
        [monthly_frequency, monthly_max_intensity, monthly_max_duration],
        axis=1
    ).sort_index()

    monthly_df.index = pd.to_datetime(monthly_df.index)
    monthly_df.index.name = "month_start"

    return monthly_df, event_df

In [2]:
month_df, event_df = extract_monthly_event_stats_fast('~/signal-extraction/vae_input_data/tp_mon-sum.nc')

In [4]:
event_df

,latitude,longitude,event_id,month_start,start_time,end_time,duration_days,tp_m
0,40.00,-123.00,1,1940-01-01,1940-01-16 00:00:00,1940-07-16 00:00:00,7,0.889413
3386,39.25,-122.00,1,1940-01-01,1940-01-16 00:00:00,1940-06-15 12:00:00,6,0.515670
16874,36.75,-119.75,1,1940-01-01,1940-01-16 00:00:00,1940-06-15 12:00:00,6,0.399136
16995,36.75,-119.50,1,1940-01-01,1940-01-16 00:00:00,1940-06-15 12:00:00,6,0.567051
3277,39.25,-122.25,1,1940-01-01,1940-01-16 00:00:00,1940-06-15 12:00:00,6,0.594467
...,...,...,...,...,...,...,...,...
20506,36.00,-121.50,121,2024-11-01,2024-11-15 12:00:00,2025-04-15 12:00:00,6,0.341987
17595,36.50,-121.25,119,2024-11-01,2024-11-15 12:00:00,2025-05-16 00:00:00,7,0.237762
14944,37.00,-121.50,112,2024-11-01,2024-11-15 12:00:00,2025-05-16 00:00:00,7,0.300345
19082,36.25,-121.25,123,2024-11-01,2024-11-15 12:00:00,2025-04-15 12:00:00,6,0.259835


In [5]:
import xarray as xr
import pandas as pd
import numpy as np
from scipy.ndimage import label, generate_binary_structure


def extract_study_area_event_stats(
    filepath: str,
    precip_var: str = "tp",
    time_dim: str = "valid_time",
    threshold_m: float = 0.001,
    connectivity: int = 1,
):
    """
    Label spatially and temporally contiguous precipitation events over a study area.

    Parameters
    ----------
    filepath : str
        Path to NetCDF file.
    precip_var : str, default "tp"
        Name of precipitation variable.
    time_dim : str, default "valid_time"
        Name of time dimension.
    threshold_m : float, default 0.001
        Wet threshold in meters.
    connectivity : int, default 1
        Connectivity passed to scipy.ndimage.generate_binary_structure.
        1 = face-connected only (recommended default).
        2 or 3 allow more diagonal connectivity.

    Returns
    -------
    monthly_df : pd.DataFrame
        Monthly DataFrame indexed by month-start datetime with columns:
        - event_frequency
        - max_intensity_m
        - max_duration_days
    event_df : pd.DataFrame
        Event-level statistics for the whole study area.
    labeled_da : xr.DataArray
        3D labeled event array with same shape as precip.
    """

    ds = xr.open_dataset(filepath)
    precip = ds[precip_var]

    if time_dim not in precip.dims:
        raise ValueError(f"{time_dim!r} is not a dimension of {precip_var!r}")

    # assume remaining dims are spatial
    spatial_dims = [d for d in precip.dims if d != time_dim]
    if len(spatial_dims) != 2:
        raise ValueError(
            f"Expected 2 spatial dims, got {spatial_dims}. "
            "This function assumes a 3D field: time x lat x lon."
        )

    # reorder to (time, y, x)
    precip = precip.transpose(time_dim, *spatial_dims)

    time_vals = pd.to_datetime(precip[time_dim].values)
    precip_vals = precip.values

    # treat NaNs as dry
    wet = np.isfinite(precip_vals) & (precip_vals > threshold_m)

    # 3D connected-component labeling
    structure = generate_binary_structure(rank=3, connectivity=connectivity)
    labels, n_events = label(wet, structure=structure)

    labeled_da = xr.DataArray(
        labels,
        coords=precip.coords,
        dims=precip.dims,
        name="event_id",
    )

    event_records = []

    for event_id in range(1, n_events + 1):
        mask = labels == event_id

        if not np.any(mask):
            continue

        # indices where event exists
        t_idx, y_idx, x_idx = np.where(mask)

        start_idx = t_idx.min()
        end_idx = t_idx.max()

        start_time = time_vals[start_idx]
        end_time = time_vals[end_idx]
        duration_days = end_idx - start_idx + 1

        # precipitation belonging to this event only
        event_precip = np.where(mask, precip_vals, 0.0)

        # event-total precip over all wet cells and all days
        event_total_precip = np.nansum(event_precip)

        # daily study-area totals for this event
        daily_area_total = np.nansum(event_precip, axis=(1, 2))

        # daily study-area mean over active event cells only
        active_counts = mask.sum(axis=(1, 2))
        with np.errstate(invalid="ignore", divide="ignore"):
            daily_area_mean = daily_area_total / active_counts
        daily_area_mean[active_counts == 0] = np.nan

        max_daily_area_total = np.nanmax(daily_area_total)
        max_daily_area_mean = np.nanmax(daily_area_mean)
        max_area_cells = np.max(active_counts)

        month_start = pd.Timestamp(start_time).to_period("M").to_timestamp()

        event_records.append(
            {
                "event_id": event_id,
                "month_start": month_start,
                "start_time": start_time,
                "end_time": end_time,
                "duration_days": duration_days,
                "event_total_precip_m": event_total_precip,
                "max_daily_area_total_m": max_daily_area_total,
                "max_daily_area_mean_m": max_daily_area_mean,
                "max_area_cells": max_area_cells,
            }
        )

    event_df = pd.DataFrame(event_records)

    if event_df.empty:
        monthly_df = pd.DataFrame(
            columns=["event_frequency", "max_intensity_m", "max_duration_days"]
        )
        monthly_df.index = pd.DatetimeIndex([], name="month_start")
        return monthly_df, event_df, labeled_da

    event_df = event_df.sort_values("start_time")

    # choose intensity metric here
    monthly_frequency = event_df.groupby("month_start").size()
    monthly_frequency.name = "event_frequency"

    monthly_max_intensity = event_df.groupby("month_start")["event_total_precip_m"].max()
    monthly_max_intensity.name = "max_intensity_m"

    monthly_max_duration = event_df.groupby("month_start")["duration_days"].max()
    monthly_max_duration.name = "max_duration_days"

    monthly_df = pd.concat(
        [monthly_frequency, monthly_max_intensity, monthly_max_duration],
        axis=1,
    ).sort_index()

    monthly_df.index = pd.to_datetime(monthly_df.index)
    monthly_df.index.name = "month_start"

    return monthly_df, event_df, labeled_da

In [6]:
monthly_df, event_df, labeled_da = extract_study_area_event_stats('~/signal-extraction/vae_input_data/tp_mon-sum.nc')